# AvitoTech ML CUP 2026 — one neural recommender

Standalone Kaggle notebook. Run all cells; the final artifact is `submission.csv` in the current working directory. All paths and resource limits are in the first code cell.

In [ ]:
# =========================
# 0. CONFIG: edit here only
# =========================
from pathlib import Path

# Kaggle dataset root from the starter notebook output.
DATA_ROOT = Path('/kaggle/input/datasets/nikitakuznetsof/avito-ml-cup-2026')

# Output file name exactly as requested: no absolute root path.
SUBMISSION_FILE = 'submission.csv'
MODEL_DIR = Path('model_artifacts')
MODEL_DIR.mkdir(exist_ok=True)

# Memory/speed controls. These defaults are intended for a ~40 minute Kaggle GPU run.
SEED = 42
MAX_SEQ_LEN = 64
MAX_TRAIN_POSITIVES = 700_000       # increase to 1_500_000+ if RAM/time allows
CANDIDATE_POOL_SIZE = 4096          # neural scoring pool; increase for quality if time allows
BATCH_SIZE = 1024
EPOCHS = 4
NEGATIVES = 48
EMB_DIM = 128
HIDDEN_DIM = 192
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2

# Keep False for RAM safety. Eval-user history is enough to train a sequence next-contact model.
# Turning this on reads extra train partitions and may be heavy.
USE_TRAIN_PARTITIONS = False
TRAIN_PARTITIONS_GLOB = 'train_080-099/train_data/part_*.parquet'
MAX_EXTRA_TRAIN_FILES = 4

print('DATA_ROOT:', DATA_ROOT)
print('SUBMISSION_FILE:', SUBMISSION_FILE)


In [ ]:
# =========================
# 1. Imports and environment
# =========================
import gc
import json
import math
import os
import random
import zipfile
from collections import Counter, deque
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm.auto import tqdm

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)
if DEVICE.type == 'cuda':
    print(torch.cuda.get_device_name(0))


In [ ]:
# =========================
# 2. Path helpers
# =========================
def must_exist(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def find_first(patterns: Iterable[str]) -> Path:
    for pattern in patterns:
        hits = sorted(DATA_ROOT.glob(pattern))
        if hits:
            return hits[0]
    raise FileNotFoundError(f'Could not find any of: {list(patterns)}')

ITEMS_PATH = must_exist(DATA_ROOT / 'item_features.parquet')
EVAL_USERS_PATH = must_exist(DATA_ROOT / 'eval_users.csv')
CONTACT_EIDS_PATH = must_exist(DATA_ROOT / 'contact_eids.csv')
EVAL_EVENTS_PATH = find_first(['eval_user_events/eval_user_events.pq', 'eval_user_events.pq', 'eval_user_events.parquet'])

print('ITEMS_PATH:', ITEMS_PATH)
print('EVAL_EVENTS_PATH:', EVAL_EVENTS_PATH)
print('EVAL_USERS_PATH:', EVAL_USERS_PATH)
print('CONTACT_EIDS_PATH:', CONTACT_EIDS_PATH)


In [ ]:
# =========================
# 3. Load item metadata and build categorical feature matrix
# =========================
ITEM_ID_COL = 'item_id'
PREFERRED_FEATURES = [
    'vertical_id', 'category_ext_y', 'region_id_y', 'loc_id_y',
    'sid_0_y', 'sid_1_y', 'sid_2_y', 'sid_3_y',
]

schema_names = pq.ParquetFile(ITEMS_PATH).schema.names
feature_cols = [c for c in PREFERRED_FEATURES if c in schema_names]
if not feature_cols:
    # Fallback for schema changes: use a compact set of non-text columns.
    feature_cols = [c for c in schema_names if c != ITEM_ID_COL and not any(x in c.lower() for x in ['title', 'description', 'text', 'embedding', 'vector'])][:12]

items = pd.read_parquet(ITEMS_PATH, columns=[ITEM_ID_COL] + feature_cols)
items = items.drop_duplicates(ITEM_ID_COL).reset_index(drop=True)
items[ITEM_ID_COL] = items[ITEM_ID_COL].astype('int64')
print('items:', items.shape)
print('feature_cols:', feature_cols)

item_ids = items[ITEM_ID_COL].to_numpy(np.int64)
item_to_row = {int(item_id): i + 1 for i, item_id in enumerate(item_ids)}  # 0 is PAD/unknown
row_to_item_id = np.concatenate([np.array([-1], dtype=np.int64), item_ids])

feature_matrix = np.zeros((len(items) + 1, len(feature_cols)), dtype=np.int32)
cardinalities = []
for j, col in enumerate(feature_cols):
    values = items[col].fillna('__NA__').astype(str)
    uniques = pd.Index(values.unique())
    vocab = {v: i + 1 for i, v in enumerate(uniques)}
    feature_matrix[1:, j] = values.map(vocab).fillna(0).astype('int32').to_numpy()
    cardinalities.append(len(vocab))

print('feature_matrix:', feature_matrix.shape)
print('cardinalities:', cardinalities)
del items
gc.collect()


In [ ]:
# =========================
# 4. Load eval users and contact event ids
# =========================
eval_users_df = pd.read_csv(EVAL_USERS_PATH)
USER_COL = 'user_id' if 'user_id' in eval_users_df.columns else eval_users_df.columns[0]
eval_users = eval_users_df[USER_COL].astype('int64').to_numpy()
eval_user_set = set(map(int, eval_users))
print('eval users:', len(eval_users))

contact_df = pd.read_csv(CONTACT_EIDS_PATH)
CONTACT_COL = 'mapped_eid' if 'mapped_eid' in contact_df.columns else contact_df.columns[0]
contact_eids = set(contact_df[CONTACT_COL].astype(int).tolist())
print('contact_eids:', sorted(contact_eids))


In [ ]:
# =========================
# 5. Streaming event pass: build histories, training examples, candidate stats
# =========================
EVENT_COLUMNS = ['timestamp', 'eid', 'user_id', 'item_id']

def iter_event_batches(paths: List[Path], batch_size: int = 250_000):
    for path in paths:
        pf = pq.ParquetFile(path)
        cols = [c for c in EVENT_COLUMNS if c in pf.schema.names]
        missing = set(EVENT_COLUMNS) - set(cols)
        if missing:
            raise ValueError(f'{path} is missing columns: {missing}')
        for batch in pf.iter_batches(batch_size=batch_size, columns=EVENT_COLUMNS):
            df = batch.to_pandas()
            yield df
            del df
            gc.collect()

event_paths = [EVAL_EVENTS_PATH]
if USE_TRAIN_PARTITIONS:
    extra = sorted(DATA_ROOT.glob(TRAIN_PARTITIONS_GLOB))[:MAX_EXTRA_TRAIN_FILES]
    event_paths = extra + event_paths
print('event files:', [str(p) for p in event_paths])

histories: Dict[int, deque] = {int(u): deque(maxlen=MAX_SEQ_LEN) for u in eval_users}
item_counts = Counter()
eid_counts = Counter()

seq_items_list: List[np.ndarray] = []
seq_eids_list: List[np.ndarray] = []
pos_items_list: List[int] = []

seen_events = 0
used_events = 0
unknown_items = 0

for batch_df in tqdm(iter_event_batches(event_paths), desc='stream events'):
    # Sorting within a batch improves sequence order without global RAM-heavy sort.
    batch_df = batch_df.sort_values(['user_id', 'timestamp'])
    for ts, eid, user_id, item_id in batch_df[EVENT_COLUMNS].itertuples(index=False, name=None):
        seen_events += 1
        user_id = int(user_id)
        if user_id not in eval_user_set:
            continue
        row = item_to_row.get(int(item_id), 0)
        if row == 0:
            unknown_items += 1
            continue
        used_events += 1
        eid = int(eid)
        h = histories[user_id]
        item_counts[row] += 1
        eid_counts[eid] += 1
        if eid in contact_eids and len(h) > 0 and len(pos_items_list) < MAX_TRAIN_POSITIVES:
            arr_i = np.zeros(MAX_SEQ_LEN, dtype=np.int32)
            arr_e = np.zeros(MAX_SEQ_LEN, dtype=np.int16)
            hist = list(h)[-MAX_SEQ_LEN:]
            offset = MAX_SEQ_LEN - len(hist)
            if hist:
                arr_i[offset:] = [x[0] for x in hist]
                arr_e[offset:] = [x[1] for x in hist]
            seq_items_list.append(arr_i)
            seq_eids_list.append(arr_e)
            pos_items_list.append(row)
        h.append((row, min(eid, 32767)))
    if len(pos_items_list) >= MAX_TRAIN_POSITIVES:
        # Continue scanning eval events only if extra train files were used? We still need final histories/candidates.
        pass

print('seen_events:', seen_events, 'used_eval_events:', used_events, 'unknown_items:', unknown_items)
print('training positives:', len(pos_items_list))
print('top eids:', eid_counts.most_common(10))

if not pos_items_list:
    raise RuntimeError('No training positives found. Check contact_eids and event file path.')

X_items = np.stack(seq_items_list).astype(np.int32)
X_eids = np.stack(seq_eids_list).astype(np.int16)
y = np.asarray(pos_items_list, dtype=np.int32)
print('train arrays:', X_items.shape, X_eids.shape, y.shape)

del seq_items_list, seq_eids_list, pos_items_list
gc.collect()


In [ ]:
# =========================
# 6. Lightweight EDA plots saved into reports/figures
# =========================
reports_dir = Path('reports/figures')
reports_dir.mkdir(parents=True, exist_ok=True)

if plt is not None:
    eid_s = pd.Series(dict(eid_counts)).sort_values(ascending=False).head(30).sort_values()
    colors = ['#b23a48' if int(eid) in contact_eids else '#4d6a8a' for eid in eid_s.index]
    ax = eid_s.plot(kind='barh', color=colors, figsize=(8, 7))
    ax.set_title('Top event ids in eval history')
    ax.set_xlabel('events')
    plt.tight_layout()
    plt.savefig(reports_dir / 'event_id_frequency.png', dpi=160)
    plt.close()

history_lengths = np.array([len(h) for h in histories.values()], dtype=np.int32)
summary = {
    'eval_users': int(len(eval_users)),
    'used_eval_events': int(used_events),
    'training_positives': int(len(y)),
    'candidate_pool_size': int(CANDIDATE_POOL_SIZE),
    'history_len_mean': float(history_lengths.mean()),
    'history_len_p95': float(np.quantile(history_lengths, 0.95)),
}
pd.Series(summary).to_csv('eda_summary.csv')
print(summary)


In [ ]:
# =========================
# 7. Dataset and one neural model
# =========================
class NextContactDataset(Dataset):
    def __init__(self, x_items, x_eids, y, n_items: int, negatives: int, seed: int):
        self.x_items = x_items
        self.x_eids = x_eids
        self.y = y
        self.n_items = n_items
        self.negatives = negatives
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        pos = int(self.y[idx])
        neg = self.rng.integers(1, self.n_items + 1, size=self.negatives, dtype=np.int64)
        neg[neg == pos] = (pos % self.n_items) + 1
        return {
            'seq_items': torch.from_numpy(self.x_items[idx].astype(np.int64, copy=False)),
            'seq_eids': torch.from_numpy(self.x_eids[idx].astype(np.int64, copy=False)),
            'mask': torch.from_numpy((self.x_items[idx] > 0).astype(np.float32, copy=False)),
            'pos_item': torch.tensor(pos, dtype=torch.long),
            'neg_items': torch.from_numpy(neg),
        }

class AvitoOneNeuralRecommender(nn.Module):
    def __init__(self, cardinalities, item_feature_matrix, emb_dim=128, hidden_dim=192, max_eid=512, max_seq_len=64):
        super().__init__()
        self.register_buffer('item_feature_matrix', torch.as_tensor(item_feature_matrix, dtype=torch.long))
        self.feature_embeddings = nn.ModuleList([
            nn.Embedding(int(card) + 1, emb_dim, padding_idx=0) for card in cardinalities
        ])
        self.item_norm = nn.LayerNorm(emb_dim)
        self.item_proj = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.GELU(), nn.LayerNorm(emb_dim))
        self.eid_embedding = nn.Embedding(int(max_eid) + 1, emb_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_seq_len, emb_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=4,
            dim_feedforward=hidden_dim * 4,
            dropout=0.1,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.user_proj = nn.Sequential(nn.LayerNorm(emb_dim), nn.Linear(emb_dim, emb_dim), nn.Tanh())
        self.item_bias = nn.Embedding(item_feature_matrix.shape[0], 1)
        self.temperature = nn.Parameter(torch.tensor(0.07))

    def item_encode(self, item_rows):
        flat = item_rows.reshape(-1).clamp_min(0)
        feats = self.item_feature_matrix[flat]
        emb = 0
        for j, layer in enumerate(self.feature_embeddings):
            emb = emb + layer(feats[:, j])
        emb = self.item_proj(self.item_norm(emb))
        return emb.reshape(*item_rows.shape, -1)

    def user_encode(self, seq_items, seq_eids, mask):
        item_emb = self.item_encode(seq_items.clamp_min(0))
        eid_emb = self.eid_embedding(seq_eids.clamp(0, self.eid_embedding.num_embeddings - 1))
        pos = torch.arange(seq_items.shape[1], device=seq_items.device).unsqueeze(0)
        x = (item_emb + eid_emb + self.pos_embedding(pos)) * mask.unsqueeze(-1)
        key_padding = mask.eq(0)
        empty = mask.sum(1).eq(0)
        if empty.any():
            key_padding = key_padding.clone()
            key_padding[empty] = False
        enc = self.encoder(x, src_key_padding_mask=key_padding)
        pooled = (enc * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1.0)
        return self.user_proj(pooled)

    def score_items(self, user_vec, item_rows):
        item_vec = self.item_encode(item_rows)
        scores = (user_vec.unsqueeze(-2) * item_vec).sum(-1)
        scores = scores / self.temperature.abs().clamp_min(0.02)
        scores = scores + self.item_bias(item_rows).squeeze(-1)
        return scores

    def forward(self, batch):
        user_vec = self.user_encode(batch['seq_items'], batch['seq_eids'], batch['mask'])
        candidates = torch.cat([batch['pos_item'].unsqueeze(1), batch['neg_items']], dim=1)
        return self.score_items(user_vec, candidates)


In [ ]:
# =========================
# 8. Train
# =========================
n_items = len(row_to_item_id) - 1
max_eid = max(max(eid_counts.keys()) if eid_counts else 1, max(contact_eids) if contact_eids else 1)
print('n_items:', n_items, 'max_eid:', max_eid)

perm = np.random.default_rng(SEED).permutation(len(y))
X_items = X_items[perm]
X_eids = X_eids[perm]
y = y[perm]

dataset = NextContactDataset(X_items, X_eids, y, n_items=n_items, negatives=NEGATIVES, seed=SEED)
val_size = max(1, min(len(dataset) // 20, 50_000))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')

model = AvitoOneNeuralRecommender(
    cardinalities=cardinalities,
    item_feature_matrix=feature_matrix,
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    max_eid=max_eid,
    max_seq_len=MAX_SEQ_LEN,
).to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')
best_val = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for batch in tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}'):
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        labels = torch.zeros(batch['pos_item'].shape[0], dtype=torch.long, device=DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            logits = model(batch)
            loss = F.cross_entropy(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        losses.append(float(loss.detach().cpu()))

    model.eval()
    val_losses, val_top1 = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            labels = torch.zeros(batch['pos_item'].shape[0], dtype=torch.long, device=DEVICE)
            logits = model(batch)
            val_losses.append(float(F.cross_entropy(logits, labels).cpu()))
            val_top1.append(float((logits.argmax(1) == 0).float().mean().cpu()))
    val_loss = float(np.mean(val_losses))
    top1 = float(np.mean(val_top1))
    print(f'epoch={epoch} train_loss={np.mean(losses):.5f} val_loss={val_loss:.5f} val_top1={top1:.5f}')
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), MODEL_DIR / 'model.pt')

model.load_state_dict(torch.load(MODEL_DIR / 'model.pt', map_location=DEVICE))
print('best_val_loss:', best_val)


In [ ]:
# =========================
# 9. Build neural candidate pool
# =========================
# Candidate generation is deliberately simple and non-trainable. Final ranking is produced by the one neural model.
# This avoids a RAM/time explosion from scoring every catalog item for every eval user.
most_common_rows = [row for row, _ in item_counts.most_common(CANDIDATE_POOL_SIZE * 2)]

# Add a broad vertical/category tail if popularity alone is too short.
all_rows = np.arange(1, len(row_to_item_id), dtype=np.int64)
pool = []
seen = set()
for row in most_common_rows:
    if row not in seen:
        pool.append(int(row)); seen.add(int(row))
for row in all_rows:
    if len(pool) >= CANDIDATE_POOL_SIZE:
        break
    if int(row) not in seen:
        pool.append(int(row)); seen.add(int(row))

candidate_rows_np = np.asarray(pool[:CANDIDATE_POOL_SIZE], dtype=np.int64)
if len(candidate_rows_np) < 160:
    raise RuntimeError(f'Candidate pool too small: {len(candidate_rows_np)}')
print('candidate pool:', len(candidate_rows_np))


In [ ]:
# =========================
# 10. Predict and save submission.csv
# =========================
def make_user_tensors(user_ids: np.ndarray):
    seq_i = np.zeros((len(user_ids), MAX_SEQ_LEN), dtype=np.int64)
    seq_e = np.zeros((len(user_ids), MAX_SEQ_LEN), dtype=np.int64)
    masks = np.zeros((len(user_ids), MAX_SEQ_LEN), dtype=np.float32)
    seen_sets = []
    for n, user_id in enumerate(user_ids):
        h = list(histories.get(int(user_id), []))[-MAX_SEQ_LEN:]
        seen_sets.append({int(x[0]) for x in histories.get(int(user_id), [])})
        if h:
            off = MAX_SEQ_LEN - len(h)
            seq_i[n, off:] = [x[0] for x in h]
            seq_e[n, off:] = [x[1] for x in h]
            masks[n, off:] = 1.0
    return seq_i, seq_e, masks, seen_sets

model.eval()
rows_out = []
candidate_rows = torch.tensor(candidate_rows_np, dtype=torch.long, device=DEVICE)

with torch.no_grad():
    # Encode candidate items once. This is the key memory/time saver.
    candidate_vec = model.item_encode(candidate_rows)
    candidate_bias = model.item_bias(candidate_rows).squeeze(-1)
    temp = model.temperature.abs().clamp_min(0.02)

    for start in tqdm(range(0, len(eval_users), BATCH_SIZE), desc='predict users'):
        user_batch = eval_users[start:start + BATCH_SIZE]
        seq_i, seq_e, masks, seen_sets = make_user_tensors(user_batch)
        seq_i = torch.tensor(seq_i, dtype=torch.long, device=DEVICE)
        seq_e = torch.tensor(seq_e, dtype=torch.long, device=DEVICE)
        masks = torch.tensor(masks, dtype=torch.float32, device=DEVICE)
        user_vec = model.user_encode(seq_i, seq_e, masks)
        scores = (user_vec @ candidate_vec.T) / temp + candidate_bias.unsqueeze(0)

        # Do not recommend items already seen in the user's history.
        for i, seen_rows in enumerate(seen_sets):
            if not seen_rows:
                continue
            hit_idx = np.flatnonzero(np.isin(candidate_rows_np, np.fromiter(seen_rows, dtype=np.int64)))
            if len(hit_idx):
                scores[i, torch.tensor(hit_idx, dtype=torch.long, device=DEVICE)] = -1e30

        _, idx = torch.topk(scores, k=160, dim=1)
        pred_rows = candidate_rows[idx].cpu().numpy()
        for user_id, pred in zip(user_batch, pred_rows):
            item_pred = row_to_item_id[pred]
            rows_out.extend((int(user_id), int(item_id)) for item_id in item_pred)

submission = pd.DataFrame(rows_out, columns=['user_id', 'item_id']).drop_duplicates(['user_id', 'item_id'])
counts = submission.groupby('user_id').size()
print('submission shape:', submission.shape)
print('users:', submission['user_id'].nunique(), 'min_k:', int(counts.min()), 'max_k:', int(counts.max()))
assert list(submission.columns) == ['user_id', 'item_id']
assert submission.duplicated(['user_id', 'item_id']).sum() == 0
assert counts.max() <= 160
submission.to_csv(SUBMISSION_FILE, index=False)
print('saved:', SUBMISSION_FILE)


## Notes

This notebook uses exactly one trainable model: `AvitoOneNeuralRecommender`. Candidate pooling is a non-trainable memory control step; final scores and final ranking are produced by the neural network.